In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/clean_emails.csv")

# Remove empty or NaN text rows
df = df.dropna(subset=["clean_text"])
df = df[df["clean_text"].str.strip() != ""]

print(df.shape)
df.head()


(29832, 4)


,text,label,source,clean_text
0,"hpl nom for may 25 , 2001 ( see attached file ...",0,enron,hpl nom for may see attached file hplno xls hp...
1,re : nom / actual vols for 24 th - - - - - - -...,0,enron,re nom actual vols for th forwarded by sabrae ...
2,"enron actuals for march 30 - april 1 , 201 est...",0,enron,enron actuals for march april estimated actual...
3,"hpl nom for june 1 , 2001 ( see attached file ...",0,enron,hpl nom for june see attached file hplno xls h...
4,# 9760 tried to get fancy with your address an...,0,enron,tried to get fancy with your address and it ca...


In [2]:
from sklearn.model_selection import train_test_split

X = df["clean_text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


Training samples: 23865
Testing samples: 5967


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("Vector shape:", X_train_vec.shape)


Vector shape: (23865, 5000)


In [4]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

print("Model trained successfully")


Model trained successfully


In [5]:
from sklearn.metrics import classification_report, accuracy_score

y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.9842466901290431
              precision    recall  f1-score   support

           0       0.99      0.98      0.98      2877
           1       0.98      0.99      0.98      3090

    accuracy                           0.98      5967
   macro avg       0.98      0.98      0.98      5967
weighted avg       0.98      0.98      0.98      5967



In [6]:
import joblib
import os

# Create models folder if not exists
os.makedirs("../outputs/models", exist_ok=True)

# Save model and vectorizer
joblib.dump(model, "../outputs/models/logistic_model.joblib")
joblib.dump(vectorizer, "../outputs/models/tfidf_vectorizer.joblib")

print("Model and vectorizer saved successfully ✅")


Model and vectorizer saved successfully ✅


In [7]:
import joblib

# Load saved model and vectorizer
model_loaded = joblib.load("../outputs/models/logistic_model.joblib")
vectorizer_loaded = joblib.load("../outputs/models/tfidf_vectorizer.joblib")

print("Model loaded successfully ✅")


Model loaded successfully ✅


In [8]:
sample_email = "Urgent! Your bank account is suspended. Click this link to verify immediately."

sample_vec = vectorizer_loaded.transform([sample_email])
prediction = model_loaded.predict(sample_vec)

if prediction[0] == 1:
    print("🚨 Prediction: PHISHING")
else:
    print("✅ Prediction: LEGITIMATE")


🚨 Prediction: PHISHING


In [9]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

# Train SVM model
svm_model = LinearSVC()
svm_model.fit(X_train_vec, y_train)

# Predict
y_pred_svm = svm_model.predict(X_test_vec)

# Evaluate
print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))


SVM Accuracy: 0.9865929277693983
              precision    recall  f1-score   support

           0       0.99      0.98      0.99      2877
           1       0.98      0.99      0.99      3090

    accuracy                           0.99      5967
   macro avg       0.99      0.99      0.99      5967
weighted avg       0.99      0.99      0.99      5967

